[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fiit-ba/zneus-2026/blob/main/labs/week_08_transfer_learning/task_8_pets_transfer_learning.ipynb)

# Week 8 · Transfer learning: ResNet on Oxford-IIIT Pets

Recognise the breed of a cat or dog from a photo: 37 breeds, about 100 training photos per breed. Start from a model pretrained on ImageNet (`torchvision.models`) and adapt it to the 37 breeds. Metric: accuracy.

Working in Colab? Replace `fiit-ba` in the badge URL with your GitHub username to open the copy in your fork, and run the setup cell below. **A GPU is recommended** (Colab: *Runtime -> Change runtime type -> T4 GPU*); on a laptop develop with `SUBSET` set.

In [ ]:
# Colab setup (does nothing when you run locally)
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import wandb

# --- Configuration ---
SEED = 42
SUBSET = None        # e.g. 256 while debugging: train on only that many rows (see labs/README.md); None = everything
DATA_DIR = Path(os.environ.get("ZNEUS_DATA_DIR", "data"))

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

## Data

Download and unzip the competition data from the *Data* tab of the Kaggle competition (link given at the lab) into `data/kaggle/` next to this notebook (or into `$ZNEUS_DATA_DIR/kaggle/`), so that the images are at `data/kaggle/train/<id>.jpg` and `data/kaggle/test/<id>.jpg`. With the Kaggle CLI: `uv run kaggle competitions download -c <competition-slug> -p data/kaggle` and unzip there (about 200 MB).

| file | content |
|---|---|
| `train/<id>.jpg` + `train.csv` | 3 680 training photos; `train.csv` has the columns `id,breed` |
| `test/<id>.jpg` + `sample_submission.csv` | 3 669 test photos, **no labels**; `sample_submission.csv` lists the ids and shows the required format `id,breed` |

JPEG, RGB, shorter side 256 px, varying aspect ratio. The `breed` is one of the 37 strings in `CLASSES` below, exactly as spelled there.

The cell below builds `train_df` (`id`, `path`, `breed`, `label` = index into `CLASSES`) and `test_df` (`id`, `path`). Without the Kaggle files it falls back to the torchvision copy of the Oxford-IIIT Pet dataset (800 MB download), so the notebook runs anywhere; the fallback test ids are not the Kaggle ids, so that `submission.csv` will not score on Kaggle.

In [ ]:
CLASSES = ["Abyssinian", "American Bulldog", "American Pit Bull Terrier", "Basset Hound", "Beagle", "Bengal", "Birman", "Bombay",
           "Boxer", "British Shorthair", "Chihuahua", "Egyptian Mau", "English Cocker Spaniel", "English Setter", "German Shorthaired",
           "Great Pyrenees", "Havanese", "Japanese Chin", "Keeshond", "Leonberger", "Maine Coon", "Miniature Pinscher", "Newfoundland",
           "Persian", "Pomeranian", "Pug", "Ragdoll", "Russian Blue", "Saint Bernard", "Samoyed", "Scottish Terrier", "Shiba Inu",
           "Siamese", "Sphynx", "Staffordshire Bull Terrier", "Wheaten Terrier", "Yorkshire Terrier"]
KAGGLE_DIR = DATA_DIR / "kaggle"

if all((KAGGLE_DIR / p).exists() for p in ["train.csv", "sample_submission.csv", "train", "test"]):
    train_df = pd.read_csv(KAGGLE_DIR / "train.csv")
    train_df["path"] = [KAGGLE_DIR / "train" / f"{i}.jpg" for i in train_df["id"]]
    test_df = pd.read_csv(KAGGLE_DIR / "sample_submission.csv")[["id"]]
    test_df["path"] = [KAGGLE_DIR / "test" / f"{i}.jpg" for i in test_df["id"]]
    y_test_local = None
    print("Kaggle files found in", KAGGLE_DIR)
else:
    from torchvision.datasets import OxfordIIITPet

    PETS_DIR = DATA_DIR / "oxford-iiit-pet"
    for split in ("trainval", "test"):  # downloads images/ and annotations/ into PETS_DIR when missing
        OxfordIIITPet(DATA_DIR, split=split, download=True)

    def read_pets_split(split: str):
        """annotations/<split>.txt has one line per image: '<image_id> <breed id 1..37> <species> <breed id in species>'."""
        rows = [line.split() for line in (PETS_DIR / "annotations" / f"{split}.txt").read_text().splitlines() if line.strip()]
        image_ids = [row[0] for row in rows]
        labels = np.array([int(row[1]) - 1 for row in rows])
        paths = [PETS_DIR / "images" / f"{image_id}.jpg" for image_id in image_ids]
        return image_ids, paths, labels

    train_ids, train_paths, train_labels = read_pets_split("trainval")
    _, test_paths, y_test_local = read_pets_split("test")
    # breed name = image id without its number, e.g. "american_pit_bull_terrier_12" -> "American Pit Bull Terrier"
    breed_of_label = {l: " ".join(part.title() for part in i.rsplit("_", 1)[0].split("_")) for i, l in zip(train_ids, train_labels)}
    assert [breed_of_label[l] for l in range(len(CLASSES))] == CLASSES, "breed names in the annotations differ from CLASSES"
    train_df = pd.DataFrame({"id": np.arange(len(train_paths)), "path": train_paths, "breed": [CLASSES[l] for l in train_labels]})
    test_df = pd.DataFrame({"id": np.arange(len(test_paths)), "path": test_paths})
    print(f"No Kaggle files in {KAGGLE_DIR} -> torchvision Oxford-IIIT Pet (its test ids are NOT the Kaggle ids)")

assert set(train_df["breed"]) <= set(CLASSES), "unexpected breed names in train.csv"
train_df["label"] = train_df["breed"].map(CLASSES.index)
# the files are sorted by breed; shuffle once (seeded) so that a SUBSET of the first rows still covers all breeds
train_df = train_df.iloc[np.random.default_rng(SEED).permutation(len(train_df))].reset_index(drop=True)
if SUBSET is not None:
    train_df = train_df.head(SUBSET)

print(f"train_df: {len(train_df)} rows {list(train_df.columns)}   test_df: {len(test_df)} rows {list(test_df.columns)}   {len(CLASSES)} breeds")

## Your task

Fine-tune an ImageNet-pretrained model from `torchvision.models` to the 37 breeds as well as you can, log every run to Weights & Biases (project `zneus-2026`, run names `week08-...`) and submit your test predictions to Kaggle. Your code goes into the cell below; it must end with `test_pred` (a numpy array, a list or a torch tensor), one predicted breed **index** (into `CLASSES`) per row of `test_df`, in the same order. The last cell maps the indices to breed names and writes `submission.csv`.

`wandb login` once in a terminal (API key from https://wandb.ai/authorize); without an account set `WANDB_MODE=offline` and `wandb sync` the runs later. Set the project `zneus-2026` to **Public** before you hand in.

In [ ]:
# TODO: your solution. When this cell has run, `test_pred` must hold one predicted breed index per row of test_df.
test_pred = ...

## Kaggle submission

`submission.csv` needs exactly the columns `id,breed` with the ids of `sample_submission.csv` and the breed **name** spelled as in `CLASSES` (case and spaces matter), no index column. Upload it on the Kaggle competition page (*Submit Predictions*) or with `uv run kaggle competitions submit -c <competition-slug> -f submission.csv -m "week 8"`.

In [ ]:
assert test_pred is not ..., "fill in the solution cell above: test_pred is still `...`"
if torch.is_tensor(test_pred):
    test_pred = test_pred.detach().cpu().numpy()
test_pred = np.asarray(test_pred).reshape(-1).astype(int)
assert len(test_pred) == len(test_df), f"test_pred has {len(test_pred)} values, expected one per test image ({len(test_df)})"
assert ((test_pred >= 0) & (test_pred < len(CLASSES))).all(), "predictions must be breed indices 0..36"

submission = pd.DataFrame({"id": test_df["id"].to_numpy(), "breed": [CLASSES[i] for i in test_pred]})
submission.to_csv("submission.csv", index=False)
print(f"wrote submission.csv: {len(submission)} rows, columns {list(submission.columns)}, {submission['breed'].nunique()} distinct breeds")
if y_test_local is not None:
    local_accuracy = (test_pred == y_test_local).mean()
    print(f"accuracy on the torchvision test split: {local_accuracy:.4f}")
submission.head()

## Before you hand in

- [ ] the notebook runs top to bottom and is committed and pushed to your fork (`data/`, `wandb/` and `submission.csv` are git-ignored, leave them out),
- [ ] `submission.csv` is on the Kaggle competition leaderboard under your AIS login,
- [ ] your W&B project `zneus-2026` is public and contains your runs; hand in the link.